# Predictive Maintenance for a Milling Machine

**Intern:** Oluwademilade  
**Supervisor:** Joel Adebayo  
**Internship period:** October 6 to October 30, 2026  
**Repository:** https://github.com/miss-dev/Milling-machine_model.git

---

## How to use this notebook

This notebook is your project and your project documentation in one place. It follows five phases, and each phase ends with a **phase gate**: a checklist you complete before your supervisor agrees you can move on.

| Phase | When | Gate |
|---|---|---|
| 1. Define | Week 1 | Fri Oct 9 |
| 2. Design | Week 2 | Fri Oct 16 |
| 3. Build | Week 3 to Week 4 Tuesday | Tue Oct 27 |
| 4. Test and Validate | Week 4 Wednesday to Thursday | Thu Oct 29 |
| 5. Deliver | Fri Oct 30 | Final presentation |

**Rules for working in this notebook**

1. Fill in every section marked **TODO**. Replace the prompts in *italics* with your own writing.
2. Do not delete a section. If it does not apply, write "Not applicable" and one line explaining why.
3. Every plot needs a sentence underneath explaining what it shows and why it matters.
4. Write in your own words. Do not copy from other people's notebooks or from AI tools.
5. Before every commit: **Kernel → Restart & Run All**, and check that the notebook runs from top to bottom without errors.
6. Each week ends with two sections: **Key terms**, where you define the week's terms in your own words with an example from this project, and **Explore further**, where you investigate something beyond the core work.

## 0. Setup

Run this section first every time you open the notebook. Do not change the random seed: it makes your results reproducible.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import ( ConfusionMatrixDisplay, precision_recall_curve, PrecisionRecallDisplay)

RANDOM_SEED = 42

---
# Phase 1: Define (Week 1)

**Goal:** This is a project with the aim of building a machine learning model that predicts likely failure of a CNC milling machine. This model will be built using data from Matzka, S. (2020). Explainable Artificial Intelligence for Predictive Maintenance Applications. *Third International Conference on Artificial Intelligence for Industries (AI4I).

## 1.1 Problem statement

- *What machine are we studying, and what goes wrong with it?*
    The machine in question is a CNC milling machine and like any other machine it is subject to breakdown
- *Why does it matter to a factory if a failure is not predicted in time? Think about cost, safety and downtime.*
    Failure to predict machine downtime can constitute safety hazards, affect production plans and significantly affect the lifespan of the machine 
- *What exactly will the model predict, and from which inputs?*
    The model is set to predict machine failure based of it's operational data
- *Who would use this model, and how?*
    Any establishment that makes use of a milling machine. 

## 1.2 Requirements

**TODO.** Write the requirements your final model must meet. Each one must be measurable, so that in Phase 4 you can mark it as passed or failed. Two examples are filled in. Agree the target values with your supervisor.

| ID | Requirement | How it will be measured | Target |
|---|---|---|---|
| R1 | The model must catch most real failures. | Recall on the test set | *e.g. at least 0.80* |
| R2 | The model must not raise too many false alarms. | Precision on the test set | *agree with supervisor* |
| R3 | The results must be reproducible. | Restart & Run All gives the same numbers | Same results every run |

## 1.3 Dataset card

**TODO.** Complete the card. Some fields are filled in for you.

| Field | Details |
|---|---|
| Name | AI4I 2020 Predictive Maintenance Dataset |
| Source | UCI Machine Learning Repository, https://archive.ics.uci.edu/dataset/601 |
| License | Creative Commons Attribution 4.0 (CC BY 4.0) |
| Citation | Matzka, S. (2020). Explainable Artificial Intelligence for Predictive Maintenance Applications. *Third International Conference on Artificial Intelligence for Industries (AI4I)*. |
| Real or synthetic? | Machine operational data are rarely publicly assessible. According to nthe paper, the data was synthesized to reflect values from a real machine  |
| Number of rows and columns | 10,000 rows and 14 columns |
| Target variable | Machine failure  |
| Class balance | 96.61% class 0(not a failure), 3.39% class 1(failure) |
| Known limitations | class imbalanced |

**Variables.** TODO: complete the table after you have loaded the data.

| Column | Type | Unit | Meaning in plain words |
|---|---|---|---|
| UDI |int | N/A|S/N |
| Product ID | str |N/A | machine ID |
| Type | str | N/A| machine quality|
| Air temperature [K] | float 64| K | air temperature generated by random walk method |
| Process temperature [K] | float 64| K | process tempearture alsogenerated by random walk |
| Rotational speed [rpm] | int | RPM| machine speed|
| Torque [Nm] | float 64| Nm |machine torque |
| Tool wear [min] |int |minutes | tool wear |
| Machine failure |int | N/A| Target feature|
| TWF, HDF, PWF, OSF, RNF | int |N/A |Specific failure modes |

## 1.4 Load the data

This cell first tries to load a local copy from `data/ai4i2020.csv`. If there is none, it downloads the dataset and saves a copy, so you only download it once.

Install the downloader once with: `pip install ucimlrepo`

In [3]:
df_raw = pd.read_csv("/Users/demilade/Documents/Milling-machine_model/ai4i2020.csv")
df_raw.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


## 1.5 Data audit (Week 1 deliverable)

Never change `df_raw`. Work on a copy called `df`, so you can always go back to the original data.

In [4]:
df = df_raw.copy()

# shape, column names and data types
df.shape
df.info()
df.columns

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  str    
 2   Type                     10000 non-null  str    
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(3), int64(9), str(2)
me

Index(['UDI', 'Product ID', 'Type', 'Air temperature [K]',
       'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]',
       'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF',
       'RNF'],
      dtype='str')

In [5]:
# missing values in each column, and the number of duplicated rows
df.isnull().sum()
df.duplicated(keep = 'first').sum()

np.int64(0)

In [6]:
# summary statistics for the numeric columns
df.describe()

,UDI,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
count,10000.00000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000
mean,5000.50000,300.004930,310.005560,1538.776100,39.986910,107.951000,0.033900,0.004600,0.011500,0.009500,0.009800,0.00190
std,2886.89568,2.000259,1.483734,179.284096,9.968934,63.654147,0.180981,0.067671,0.106625,0.097009,0.098514,0.04355
min,1.00000,295.300000,305.700000,1168.000000,3.800000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
25%,2500.75000,298.300000,308.800000,1423.000000,33.200000,53.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
50%,5000.50000,300.100000,310.100000,1503.000000,40.100000,108.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
75%,7500.25000,301.500000,311.100000,1612.000000,46.800000,162.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
max,10000.00000,304.500000,313.800000,2886.000000,76.600000,253.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000


In [7]:
# add two new columns with the air and process temperatures in °C (°C = K - 273.15)
df['Air temperature (C)'] = df['Air temperature [K]'] - 273.15
df['Process temperature (C)'] = df['Process temperature [K]'] - 273.15

In [8]:
# how many machines failed, and what percentage is that?
average = np.mean(df['Machine failure'])
print(average)

0.0339


In [9]:
# failure count and failure rate for each product type (L, M, H). Hint: groupby
df.groupby('Type')['Machine failure'].mean()

Type
H    0.020937
L    0.039167
M    0.027694
Name: Machine failure, dtype: float64

** What did the audit show?** *Data is clean, no missing values or duplicated rows. Highly imbalanced as well. Surprisignly machine quality(low, medium, high), had very little to do with failure frequency*

## Week 1 key terms: Git, GitHub and Pandas

**TODO.** By Friday you should be able to explain every term below. Write each definition in your own words, and give an example from **this project** (a column, a plot, a line of code or a result in this notebook). At the Friday check-in your supervisor will pick three terms for you to explain out loud.

| Term | My definition (own words) |
|---|---|---|
| **Git and GitHub** | | 
| repository | A folder holding all files relating to a project | 
| clone | Collecting files from a repo to store locally on a pc | 
| commit | A snapshot of your current work| 
| staging area | Where modified files wait to be commited | 
| branch | an independent section of a repo that accepts changes without affecting the main file | 
| merge | the process of unifying changes across diff branches | 
| merge conflict | merge interruption | 
| remote | stored in the cloud| 
| push and pull | updating changes locally or remotely| 
| pull request | a request to make changes to a branch | 
| .gitignore | a file containg files got should not track | 
| README | a file typically containg a description of the project | 
| Markdown | a cell containg text | 
| **Pandas** | | 
| DataFrame | 2D array of data | 
| Series | 1D array of data | 
| index | position of an object | 
| dtype | data type of an obj(str, int 32...) | 
| loc vs. iloc | name and index based selection resp. | 
| boolean mask | | 
| groupby | categorizing data relative to the target variable | 
| aggregation | combining data info into a single result | 
| missing values (NaN) | placeholder for an empty cell | 
| vectorized operation | used to describe an operation that affects an entire column | 

## Week 1 explore further

This section is yours. Once the core work for the week is done, explore something beyond it. Pick at least one idea below, or follow your own curiosity. It does not have to work perfectly: an honest attempt with notes is worth more than a polished copy.

**Ideas to explore**

- Group tool wear into bands with `pd.cut` (e.g. 0–50, 50–100 minutes) and compare the failure rate in each band. What happens as the tool wears?
- Rewrite part of your audit using method chaining or `.query()`. Is it easier or harder to read?
- Compare how long a pandas loop takes against a vectorized operation on the same column (use `%timeit`). Why is one so much faster?
- Git: use `git log`, `git diff` and `git revert` on a test repository. How would you undo a bad commit without losing history?
- *Your own idea*

**What I explored:** 
I tried exporing which features ha dthe most impact on the target variable. I found that air and process temperature affected machine failure, while product quality did not

## Phase 1 gate checklist

Complete this before the Friday check-in. Change `[ ]` to `[x]` when done.

- [X] Problem statement answered in my own words
- [X] Requirements table has measurable targets agreed with my supervisor
- [X] Dataset card and variables table complete
- [X] Data loads and the audit runs without errors
- [X] Week 1 key terms table complete
- [X] Week 1 explore further entry written
- [X] Notebook runs with Restart & Run All
- [X] Pull request opened for `week-1-pandas`

**Phase review.** *What went well? What was hard? What is carried into the next phase?*

---
# Phase 2: Design (Week 2)

**Goal:** Understand the data well enough to make and justify decisions about features and models. Every pattern you find should be explained with engineering reasoning, not only described.

## 2.1 Univariate analysis

Look at each variable on its own: its distribution, its range and any outliers.

In [10]:
# relationship between air & process temperature in kelvin and machine failure
df.groupby('Process temperature [K]')['Machine failure'].mean()
df.groupby('Air temperature [K]')['Machine failure'].mean()

Air temperature [K]
295.3    0.000000
295.4    0.000000
295.5    0.000000
295.6    0.026316
295.7    0.055556
           ...   
304.1    0.000000
304.2    0.000000
304.3    0.000000
304.4    0.142857
304.5    0.000000
Name: Machine failure, Length: 93, dtype: float64

# relationship between air & process temperature in celcius and machine failure
df.groupby('Process temperature (C)')['Machine failure'].mean()
df.groupby('Air temperature (C)')['Machine failure'].mean()

In [11]:
# relationship between rotational speed and machine failure
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    results = df.groupby('Rotational speed [rpm]')['Machine failure'].mean().sort_values(ascending=False)
    print(results)

Rotational speed [rpm]
2886    1.000000
2676    1.000000
2663    1.000000
2659    1.000000
2270    1.000000
2639    1.000000
1257    1.000000
1259    1.000000
2633    1.000000
2617    1.000000
1996    1.000000
2579    1.000000
1267    1.000000
1272    1.000000
2567    1.000000
2564    1.000000
2563    1.000000
2549    1.000000
2153    1.000000
2497    1.000000
2478    1.000000
1181    1.000000
2097    1.000000
2672    1.000000
2372    1.000000
2874    1.000000
1229    1.000000
2833    1.000000
2678    1.000000
2825    1.000000
2760    1.000000
2737    1.000000
2721    1.000000
2861    1.000000
2710    1.000000
2709    1.000000
1200    1.000000
2706    1.000000
2271    1.000000
1238    1.000000
1235    1.000000
2691    1.000000
1307    0.666667
1262    0.500000
1299    0.500000
1285    0.500000
1284    0.500000
1236    0.500000
1212    0.500000
1256    0.500000
1222    0.500000
1271    0.500000
1258    0.500000
1227    0.500000
2636    0.500000
1243    0.500000
1263    0.400000
1270    

In [12]:
# relationship between torque and machine failure
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    results = df.groupby('Torque [Nm]')['Machine failure'].mean().sort_values()
    print(results)

Torque [Nm]
40.3    0.000000
38.4    0.000000
38.3    0.000000
38.1    0.000000
38.0    0.000000
37.6    0.000000
37.4    0.000000
37.2    0.000000
37.1    0.000000
37.0    0.000000
36.9    0.000000
36.8    0.000000
36.7    0.000000
36.6    0.000000
36.5    0.000000
36.4    0.000000
36.3    0.000000
36.2    0.000000
36.0    0.000000
35.7    0.000000
38.5    0.000000
35.6    0.000000
66.4    0.000000
38.8    0.000000
41.2    0.000000
41.1    0.000000
41.0    0.000000
40.8    0.000000
40.7    0.000000
40.6    0.000000
40.4    0.000000
40.2    0.000000
40.1    0.000000
40.0    0.000000
39.9    0.000000
39.8    0.000000
39.7    0.000000
39.6    0.000000
39.5    0.000000
39.4    0.000000
39.3    0.000000
39.2    0.000000
39.1    0.000000
38.7    0.000000
35.5    0.000000
35.3    0.000000
35.2    0.000000
32.4    0.000000
32.3    0.000000
32.2    0.000000
32.1    0.000000
32.0    0.000000
31.9    0.000000
31.8    0.000000
31.7    0.000000
31.6    0.000000
31.5    0.000000
31.4    0.000000
31

In [13]:
# relationship between tool wear and machine failure
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    results = df.groupby('Tool wear [min]')['Machine failure'].mean().sort_values(ascending= False)
    print(results)

Tool wear [min]
253    1.000000
251    1.000000
234    1.000000
246    0.666667
228    0.571429
238    0.500000
231    0.428571
216    0.384615
225    0.333333
226    0.333333
232    0.333333
222    0.307692
229    0.285714
218    0.285714
220    0.272727
235    0.250000
207    0.243243
208    0.205128
212    0.178571
233    0.166667
215    0.162162
213    0.161290
203    0.148936
227    0.142857
205    0.135135
209    0.133333
217    0.130435
210    0.128205
200    0.128205
187    0.125000
20     0.098039
206    0.097561
214    0.090909
189    0.088889
202    0.088889
194    0.086957
86     0.073171
134    0.071429
172    0.069767
191    0.068182
67     0.068182
161    0.066667
195    0.065217
153    0.065217
6      0.064516
204    0.063830
111    0.063830
101    0.062500
40     0.061224
64     0.061224
92     0.058824
219    0.058824
8      0.055556
148    0.055556
221    0.055556
182    0.052632
93     0.052632
119    0.052632
163    0.051282
149    0.050000
159    0.048780
65     0

**TODO.** *Do the ranges make physical sense for a milling machine? Which distributions are skewed, and why might that be?*

## 2.2 Relationships between variables

In [33]:
# correlation matrix of the numeric variables
rankings = df.corr(numeric_only=True)['Machine failure'].sort_values(ascending=False)

print(rankings)

Machine failure            1.000000
HDF                        0.575800
OSF                        0.531083
PWF                        0.522812
TWF                        0.362904
Torque [Nm]                0.191321
Tool wear [min]            0.105448
Air temperature (C)        0.082556
Air temperature [K]        0.082556
Process temperature (C)    0.035946
Process temperature [K]    0.035946
RNF                        0.004516
UDI                       -0.022892
Rotational speed [rpm]    -0.044188
Name: Machine failure, dtype: float64


In [ ]:
# scatter plot of torque against rotational speed

plt.figure(figsize=(10, 6))

sns.scatterplot(
    data = df, 
    x= 'Rotational speed [rpm]', 
    y= 'Torque [Nm]', 
    hue='Machine failure')

plt.show()

**TODO.** *Why do torque and rotational speed show this relationship? Hint: think about mechanical power.*

## 2.3 Failure analysis

In [ ]:
# compare the sensor values of failed and healthy machines (e.g. box plots split by "Machine failure")

In [ ]:
# count each failure mode (TWF, HDF, PWF, OSF, RNF)
print(df[['OSF', 'HDF', 'PWF', 'TWF', 'RNF']].sum().sort_values(ascending=False))

In [ ]:
# plot the class balance of "Machine failure"
df['Machine failure'].value_counts().plot(kind='bar')

## 2.4 Five findings (Week 2 deliverable)

**TODO.** Your five most important findings. Each must point to a plot above and give a mechanical explanation.

| # | Finding | Evidence (which plot) | Mechanical explanation |
|---|---|---|---|
| 1 | Torque has the most impact on failure | Not a plot, mean group aggregation | Higher twisting force directly increases physical stress on the shaft and cutting tool |
| 2 | Temperature conversions (both air and process) did not result in a better failure indicator | group aggregation | Kelvin and Celsius measure the exact same thermal energy on a shifted scale |
| 3 | Torque extremes signal likely failure | group agreggation, Torque vs. rpm scatter plot | Excessive torque snaps components, while near-zero torque indicates drive slippage or loss of load |
| 4 | Rotational speed alone is not a stong feature | group agreggation | Failures happen at both extremes (stalling at low RPM and overspeeding at high RPM), leaving a safe middle band |
| 5 | Combining rotational speed and temperature creates a stronger feature| Rotational speed vs. Temperature Difference scatter plot| Slow spindle speeds reduce airflow while hot ambient air blocks cooling, causing overheating|

## 2.5 Design decisions

**TODO.** Record each important decision before you start building. Add rows as you go, including in later phases.

| ID | Decision | Chosen | Why |
|---|---|---|---|
| D1 | Which features to engineer | Torque, Rotational speed, Tool wear, Temperature difference | Most mechanically relevant to machine failure |
| D2 | How to handle class imbalance | class_weight='balanced' and a 0.65 voting threshold| Only 3.4% of rows are failures. Class weights force the model to learn rare failures, while the 65% threshold prevents false alarms |
| D3 | Which metrics matter most | Precision as opposed to accuracy | This reveals the ability of the algorithm to detect failure better |
| D4 | Which models to compare | Tree-based classification models | These are more suited for classification problem | 

## Week 2 key terms: Exploratory Data Analysis

**TODO.** By Friday you should be able to explain every term below. Write each definition in your own words, and give an example from **this project** (a column, a plot, a line of code or a result in this notebook). At the Friday check-in your supervisor will pick three terms for you to explain out loud.

| Term | My definition (own words) |
|---|---|---|
| univariate analysis | Analysis of a single feature to see it's distribution| 
| bivariate analysis | Comparing multiple features to see the relationship between each other | 
| distribution | The spread of the values of a feature column | 
| mean | Usually for columns, an average of the values in a column | 
| median | The value at the middle position in a column, sorted from highest to lowest | 
| standard deviation | A measure of how far typical values in a column are from the average | 
| skewness | The shape of the distribution of values| 
| outlier | Values that fall outside the normal range | 
| interquartile range (IQR) | The middle 50% of the data, the gap between the 25th and 75th percentiles|
| histogram | A bar chart that shows how often a value occurs | 
| box plot | A chart showing the median, middle 50% (IQR), and outlier values |
| scatter plot | A plot showing data points between variables | 
| Pearson correlation | |
| correlation vs. causation | Two numbers moving together (correlation) does not prove one directly makes the other happen (causation)|
| class imbalance | When one category in the target feature is has a much larger occurence than the other |
| **Failure modes** | | |
| TWF (tool wear failure) | The machine fails because the cutting tool has been used for too many minutes and wears out | 
| HDF (heat dissipation failure) | The machine overheats because it spins too slowly while ambient air is too hot to cool it | 
| PWF (power failure) | The machine fails because the mechanical power (Torque × Speed) drops too low or spikes too high |  
| OSF (overstrain failure) | The machine jams because high torque is applied to an already worn-down tool | 
| RNF (random failure) | Unpredictable breakdowns that happen randomly without following any sensor threshold rule |

## Week 2 explore further

This section is yours. Once the core work for the week is done, explore something beyond it. Pick at least one idea below, or follow your own curiosity. It does not have to work perfectly: an honest attempt with notes is worth more than a polished copy.

**Ideas to explore**

- Make a seaborn `pairplot` coloured by machine failure. Which pairs of variables separate failures best?
- Look closely at tool wear failures. At what range of tool wear do they happen, and how does that match the Matzka paper?
- Compare the three product types (L, M, H). Do they behave differently, and why might a factory make products of different quality?
- Read about how real factories monitor machines (for example vibration or acoustic monitoring). What sensors would you add to this dataset if you could?
- *Your own idea*

**TODO. What I explored:** *What did you try, what did you find, and how does it connect to this week's key terms? Did it change how you think about the project?*

In [ ]:
# Week 2 exploration: your code here

## Phase 2 gate checklist

- [X] Every plot has a written explanation
- [X] Five findings table complete
- [X] Matzka (2020) paper read and failure modes understood
- [X] Design decisions recorded with reasons
- [X] Week 2 key terms table complete
- [ ] Week 2 explore further entry written
- [X] Notebook runs with Restart & Run All
- [X] Pull request opened for `week-2-eda`

**Phase review.** *What went well? What was hard? What is carried into the next phase?*

---
# Phase 3: Build (Week 3 to Week 4 Tuesday)

**Goal:** Turn the raw data into good features without leaking information, and train a set of models to compare.

## 3.1 Prevent data leakage

**TODO.** Some columns must not be given to the model. For each one, explain why.

| Column | Drop? | Reason |
|---|---|---|
| UDI | Yes| Irrelevant to the target feature |
| Product ID | Yes| Irrelevant to the target feature |
| TWF, HDF, PWF, OSF, RNF | Yes | These are the failure modes to be predicted |
| Temperature in C | Yes | Has the exact same relationship with the target feature as temperature in K |

In [24]:
TARGET = "Machine failure"

# list the columns to drop

DROP_COLUMNS = [
    "UDI",
    "Product ID",
    "TWF",
    "HDF",
    "PWF",
    "OSF",
    "RNF",
    TARGET
]
# create X (features) and y (target)
X = df.drop(columns=DROP_COLUMNS)
y = df[TARGET]


## 3.2 Train/test split

Split **before** any scaling or feature selection. The test set is locked away until Phase 4 and is used **once**. Use a stratified split, so both sets have the same failure rate.

In [25]:
# stratified split, 20% test, random_state=RANDOM_SEED
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.20, 
    stratify=y, 
    random_state=RANDOM_SEED
)

## 3.3 Feature engineering

Create features from engineering knowledge. Write each one as a function, so it can be reused and tested.

| Feature | Formula | Physical meaning |
|---|---|---|
| Mechanical power [W] | P = T × ω, where ω = 2πn / 60 | Measures the total rate of mechanical energy delivered by the rotating shaft |
| Temperature difference [K] | process temperature − air temperature | Measures the thermal gradient driving heat away from the hot machine into the cooler surrounding air |
| Strain | tool wear × torque | Measures the cumulative mechanical load acting on a progressively degrading cutting edge |

In [26]:
# create engineered features directly in both X_train and X_test

for dataset in [X_train, X_test]:
    dataset["Mechanical power [W]"] = dataset["Torque [Nm]"] * (2 * np.pi * dataset["Rotational speed [rpm]"] / 60)

    dataset["Temperature difference [K]"] = dataset["Process temperature [K]"] - dataset["Air temperature [K]"]
    
    dataset["Strain"] = dataset["Tool wear [min]"] * dataset["Torque [Nm]"]

In [27]:
# Encode "Type" (L, M, H)
X_train["Type"] = X_train["Type"].map({"L": 0, "M": 1, "H": 2})
X_test["Type"] = X_test["Type"].map({"L": 0, "M": 1, "H": 2})

## 3.4 Feature selection

Use the **training set only**.

In [28]:
# correlation of each feature with the target
print(X_train.corrwith(y_train).sort_values(ascending=False))

Torque [Nm]                   0.190034
Strain                        0.185752
Mechanical power [W]          0.175997
Tool wear [min]               0.099214
Air temperature [K]           0.086001
Air temperature (C)           0.086001
Process temperature (C)       0.040849
Process temperature [K]       0.040849
Rotational speed [rpm]       -0.039966
Type                         -0.045576
Temperature difference [K]   -0.111314
dtype: float64


In [29]:
# compare how well raw and engineered features separate failures (plots or scores)
comparison = pd.DataFrame({
    "Correlation": X_train.corrwith(y_train),
    "Mutual Info": mutual_info_classif(X_train, y_train, random_state= RANDOM_SEED)
})

## Week 3 key terms: Feature Engineering and Selection

**TODO.** By Friday you should be able to explain every term below. Write each definition in your own words, and give an example from **this project** (a column, a plot, a line of code or a result in this notebook). At the Friday check-in your supervisor will pick three terms for you to explain out loud.

| Term | My definition (own words) | 
|---|---|
| feature | Values, in this case, sensor readings that the algorithm learns from | 
| target | The value to be predicted | 
| feature engineering | creating new features with a better relationship/ correlation with the target variable | 
| domain-informed feature | A new column created using knowledge rather than raw data alone | 
| one-hot encoding | Turning a text category column into separate 0 and 1 columns so algorithms can read it without assuming rank order |
| standardization vs. normalization | Standardization centers data around 0 with a standard deviation of 1, normalization squishes all numbers strictly between 0 and 1|
| train/test split | Dividing the dataset into training data and testing data | 
| stratified split | A train/test split that locks in the exact same percentage of 0s and 1s in both piles | 
| data leakage | When values from the test set sneaks into the training data | 
| mutual information | A score measuring how much any feature helps predict the target | 
| feature importance | A score showing how heavily a trained tree model relied on each column to make its splits and predictions | 
| multicollinearity | When two or more input columns carry the exact same duplicated information, confusing linear models | 

## Week 3 explore further

This section is yours. Once the core work for the week is done, explore something beyond it. Pick at least one idea below, or follow your own curiosity. It does not have to work perfectly: an honest attempt with notes is worth more than a polished copy.

**Ideas to explore**

- Engineer a feature of your own from mechanical engineering knowledge (for example, something related to cutting energy or thermal load). Does it help?
- Try `PolynomialFeatures` to create interaction terms automatically. How do they compare with your hand-made physics features?
- Use recursive feature elimination (`RFE`) and compare the features it keeps with your own choice.
- Find out what principal component analysis (PCA) does. Would it be useful here, and what would you lose in interpretability?
- *Your own idea*

**TODO. What I explored:** *What did you try, what did you find, and how does it connect to this week's key terms? Did it change how you think about the project?*

In [30]:
# Week 3 exploration: your code here

**Week 3 checklist** (there is no phase gate this week, but complete this before the Friday check-in)

- [X] Leakage table, split and engineered features complete
- [X] Feature selection decision recorded in section 2.5
- [X] Week 3 key terms table complete
- [X] Week 3 explore further entry written
- [X] Notebook runs with Restart & Run All
- [X] Pull request opened for `week-3-features`

## 3.5 Model training

Build every model as a scikit-learn `Pipeline`, so preprocessing is fitted on training data only.

In [31]:
# decision tree

decision_tree_pipeline = Pipeline([
    ("model", DecisionTreeClassifier(max_depth=8, class_weight="balanced", random_state=RANDOM_SEED))
])

decision_tree_pipeline.fit(X_train, y_train)

decision_tree_predictions = decision_tree_pipeline.predict(X_test)

print(classification_report(y_test, decision_tree_predictions))

              precision    recall  f1-score   support

           0       0.99      0.95      0.97      1932
           1       0.34      0.78      0.48        68

    accuracy                           0.94      2000
   macro avg       0.67      0.86      0.72      2000
weighted avg       0.97      0.94      0.95      2000



In [32]:
# random forest
random_forest_pipeline = Pipeline([
    ("model", RandomForestClassifier( n_estimators=100, max_depth=8, class_weight="balanced", random_state=RANDOM_SEED))
    ])

random_forest_pipeline.fit(X_train, y_train)

random_forest_probs = random_forest_pipeline.predict_proba(X_test)[:, 1]

random_forest_preds = (random_forest_probs >= 0.65).astype(int)

print(classification_report(y_test, random_forest_preds))

print("Confusion Matrix:\n", confusion_matrix(y_test, random_forest_preds))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1932
           1       0.84      0.76      0.80        68

    accuracy                           0.99      2000
   macro avg       0.92      0.88      0.90      2000
weighted avg       0.99      0.99      0.99      2000

Confusion Matrix:
 [[1922   10]
 [  16   52]]


## 3.6 Experiment log

Record every experiment, including the ones that did not work. Use cross-validation scores on the **training set** here, not the test set.

| ID | Model | Features | Settings | CV recall | CV precision | CV F1 |
|---|---|---|---|---|---|---|---|
| E1 | Decision Tree | Raw sensors + encoded Type + Mechanical power [W], Temperature difference [K], Strain (7 features)| max_depth=8, class_weight="balanced", random_state=42, default 0.50 threshold| 78% | 34% | 48% | 
| E2 | Random Forest | Raw sensors + encoded Type + Mechanical power [W], Temperature difference [K], Strain (7 features)| n_estimators=100, max_depth=8, class_weight="balanced", random_state=42, 0.65 probability threshold| 76% | 84% | 80% | 

## Phase 3 gate checklist

- [ ] Leaky columns identified and dropped, with reasons
- [ ] Train/test split done before any fitting; test set untouched
- [ ] Engineered features written as functions
- [ ] Baseline and at least three models trained in pipelines
- [ ] Experiment log complete
- [ ] Notebook runs with Restart & Run All
- [ ] Pull request opened

**Phase review.** *What went well? What was hard? What is carried into the next phase?*

---
# Phase 4: Test and Validate (Week 4, Wednesday to Thursday)


## 4.1 Final evaluation on the test set

Run this **once**, with your final model. If you change the model after seeing these results, say so honestly in your write-up.

In [ ]:
# TODO: fit the final model on the full training set
# TODO: predict on the test set
# TODO: confusion matrix and classification report
# TODO: precision-recall curve


# Fit the final model on the full training set
random_forest_pipeline.fit(X_train, y_train)

# Predict on the test set
random_forest_probabilities = random_forest_pipeline.predict_proba(X_test)[:, 1]

random_forest_predictions = (random_forest_probabilities >= 0.65).astype(int)

# Print confusion matrix and classification report
print(classification_report(y_test, random_forest_predictions))

# Plot Confusion Matrix and Precision-Recall Curve side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    random_forest_predictions,
    display_labels=["Healthy (0)", "Failure (1)"],
    cmap="Oranges",
    ax=axes[0]
)
axes[0].set_title("Confusion Matrix (65% Threshold)")

PrecisionRecallDisplay.from_predictions(
    y_test,
    random_forest_probabilities,
    ax=axes[1]
)
axes[1].set_title("Precision-Recall Curve")

plt.tight_layout()
plt.show()


'from sklearn.metrics import (\n    classification_report, confusion_matrix, ConfusionMatrixDisplay,\n    precision_recall_curve, PrecisionRecallDisplay,\n)'

## 4.2 Requirements check

Copy your requirements from section 1.2 and record the results.

| ID | Requirement | How it will be measured | Target | Result 
|---|---|---|---|
| R1 | The model must catch most real failures. | Recall on the test set | *e.g. at least 0.80* | 0.76 
| R2 | The model must not raise too many false alarms. | Precision on the test set | At least 0.80 | 0.82 
| R3 | The results must be reproducible. | Restart & Run All gives the same numbers | Same results every run | Identical across runs

*If a requirement failed, explain why and what you would try next.*

## 4.3 Model card

**TODO.**

| Field | Details |
|---|---|
| Model | |
| Intended use | |
| Not intended for | |
| Training data | |
| Features used | |
| Test results | |
| Limitations | *e.g. trained on synthetic data from one machine type* |
| Risks | *What happens if the model misses a failure, or raises a false alarm?* |

## Week 4 key terms: Model Training and Evaluation

**TODO.** By Friday you should be able to explain every term below. Write each definition in your own words, and give an example from **this project** (a column, a plot, a line of code or a result in this notebook). At the Friday check-in your supervisor will pick three terms for you to explain out loud.

| Term | My definition (own words) | Example from this project |
|---|---|---|
| baseline model | | |
| classification | | |
| logistic regression | | |
| decision tree | | |
| random forest | | |
| overfitting and underfitting | | |
| Pipeline | | |
| confusion matrix (TP, FP, TN, FN) | | |
| precision | | |
| recall | | |
| F1-score | | |
| precision-recall curve | | |
| class weights | | |
| resampling (SMOTE) | | |
| stratified k-fold cross-validation | | |
| hyperparameter | | |
| grid search | | |
| random seed | | |
| reproducibility | | |
| model card | | |

## Week 4 explore further

This section is yours. Once the core work for the week is done, explore something beyond it. Pick at least one idea below, or follow your own curiosity. It does not have to work perfectly: an honest attempt with notes is worth more than a polished copy.

**Ideas to explore**

- Change the decision threshold of your final model (use `predict_proba`). How do precision and recall trade off as the threshold moves?
- Try a gradient boosting model (`HistGradientBoostingClassifier`) and add it to your experiment log.
- Try SMOTE from the `imbalanced-learn` package (inside the cross-validation, never before the split). Does it beat class weights?
- Instead of predicting failure or no failure, try predicting which failure mode happened. What changes?
- Use permutation importance to see which features your final model relies on. Does it agree with the physics from the Matzka paper?
- *Your own idea*

**TODO. What I explored:** *What did you try, what did you find, and how does it connect to this week's key terms? Did it change how you think about the project?*

In [ ]:
# Week 4 exploration: your code here

## Phase 4 gate checklist

- [ ] Final model chosen using cross-validation only
- [ ] Test set used once, and results reported honestly
- [ ] Requirements check complete
- [ ] Model card complete
- [ ] Week 4 key terms table complete
- [ ] Week 4 explore further entry written
- [ ] Notebook runs with Restart & Run All
- [ ] Pull request opened for `week-4-models`

**Phase review.** *What went well? What was hard? What is carried into the next phase?*

---
# Phase 5: Deliver (Friday, October 30)

**Goal:** Present the project so that someone who has not seen it can understand what you did, what you found and how to reproduce it.

## 5.1 Summary

**TODO.** *One paragraph: the problem, what you did, your final model and its key results.*

## 5.2 Limitations and future work

**TODO.** *What are the weaknesses of this project? What would you do with more time or real factory data?*

## 5.3 How to reproduce this work

**TODO.** *Python version, packages needed (with versions from section 0), and the steps to run this notebook from scratch.*

## 5.4 References

1. Matzka, S. (2020). Explainable Artificial Intelligence for Predictive Maintenance Applications. *Third International Conference on Artificial Intelligence for Industries (AI4I)*.
2. AI4I 2020 Predictive Maintenance Dataset. UCI Machine Learning Repository. https://archive.ics.uci.edu/dataset/601
3. *TODO: add any other sources you used*

## 5.5 Reflection

**TODO.** *What are the three most important things you learned? What would you do differently next time?*

## Phase 5 gate checklist

- [ ] All TODO sections filled in
- [ ] Final presentation prepared
- [ ] Notebook runs with Restart & Run All
- [ ] Final pull request merged into `main`

---
# Appendix A: Progress log

**TODO.** Write one entry per week, in your own words, before the Friday check-in.

### Week 1
- **What I did:**
- **What I learned:**
- **What confused me, and how I resolved it:**
- **Plan for next week:**

### Week 2
- **What I did:**
- **What I learned:**
- **What confused me, and how I resolved it:**
- **Plan for next week:**

### Week 3
- **What I did:**
- **What I learned:**
- **What confused me, and how I resolved it:**
- **Plan for next week:**

### Week 4
- **What I did:**
- **What I learned:**
- **What confused me, and how I resolved it:**

# Appendix B: Check-in notes

**TODO.** After each check-in, write down what was agreed.

| Date | Check-in | What was discussed | Actions agreed |
|---|---|---|---|
| Mon Oct 5 | Plan review | | |
| Wed Oct 7 | Quick check | | |
| Fri Oct 9 | Phase 1 gate | | |
| | | | |